In [1]:
from B0_MODEL import UNET
from B2_UTILS import (load_checkpoint, save_checkpoint, get_loaders,
                      check_accuracy, save_predictions_as_imgs,
                      print_and_write)
import torch
from tqdm import tqdm  # makes your loops show a smart progress meter
import torch.nn as nn
import torch.optim as optim
import os
import json
import numpy as np

# Test train? Train? Predict?
BATCH_1_TEST = False
TRAIN = False
PREDICT = True

# Set training epoch munbers, early stopping epoch num
training_epoch_num = 100
early_stopping_patience = 5

# ====================== Hyperparameters etc. ========================
MAIN_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST'
MODEL_TYPES = ['secondary_strctures', 'key_atoms', 'residue_types']
MAP_DIRS_TYPES = [['map_sample'],
                  ['map_sample', 'secondary_strctures_pred_prob'],
                  [
                      'map_sample', 'secondary_strctures_pred_prob',
                      'key_atoms_pred_prob'
                  ]]
# LOAD_MODEL = False

data_path = os.path.join(MAIN_PATH, 'train_val_data')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 8  # batch_size vs GPU Memory: 1vs2877; 2vs3485; 4vs4668; 8vs7162

num_workers = 2
learning_rate = 1e-4
pin_memory = True
# IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_DEPTH = [64, 64, 64]

In [2]:
def initial_setting(model_type, data_path, map_dirs, device):
    """
    This function performs the initial setup before training process:
    1. Read the class weights of the UNET model,
    2. Read the saved epoch number,
    3. Generate paths of file that will be loaded to UNET model.
    """

    # Load the class weights of all model types
    weight_path = os.path.join(data_path, 'class_weight_for_training.txt')
    try:
        # Load the weight for training
        with open(weight_path, "r") as file:
            weight_all = json.load(file)

        if model_type not in weight_all.keys():
            print(f"Weight for {model_type} is not found.")
            exit()
        else:
            c_w = weight_all[model_type]
            class_weights = torch.FloatTensor(c_w).to(device)

            # Calculate the number of output channels based on the number of tags in model_type
            out_channels = len(c_w)

    except FileNotFoundError:
        print("Weight file does not exist.")
        exit()

    # Load the load_epoch_num data or initialize it if the file doesn't exist
    load_epoch_num_path = os.path.join(data_path, "epoch_num_path.txt")

    if not os.path.exists(load_epoch_num_path):
        load_epoch_num_all = {model_type: 0}
    else:
        with open(load_epoch_num_path, "r") as file:
            load_epoch_num_all = json.load(file)
            if model_type not in load_epoch_num_all.keys():
                load_epoch_num_all[model_type] = 0
    load_epoch_num = load_epoch_num_all[model_type]

    # Save the updated load_epoch_num_all data
    with open(load_epoch_num_path, "w") as file:
        json.dump(load_epoch_num_all, file)

    # Calculate the number of output channels based on the num of input map types
    in_channels = len(map_dirs)

    # Set file paths and make some directories when not existed
    train_map_dirs = [
        os.path.join(data_path, 'train', map_dir) for map_dir in map_dirs
    ]
    val_map_dirs = [
        os.path.join(data_path, 'val', map_dir) for map_dir in map_dirs
    ]
    train_model_dir = os.path.join(data_path, 'train', model_type)
    val_model_dir = os.path.join(data_path, 'val', model_type)

    load_model_path = os.path.join(
        data_path, 'checkpoint', model_type,
        f'{model_type}.epoch{load_epoch_num}.pth.tar')

    evl_folder_path = os.path.join(data_path, 'evaluation', model_type)

    os.makedirs(os.path.dirname(load_model_path), exist_ok=True)
    os.makedirs(evl_folder_path, exist_ok=True)

    return class_weights, in_channels, out_channels, load_epoch_num, train_map_dirs, val_map_dirs, train_model_dir, val_model_dir, load_model_path, evl_folder_path


In [3]:
type_index = 0
model_type = MODEL_TYPES[type_index]

map_dirs = MAP_DIRS_TYPES[type_index]

class_weights, in_channels, out_channels, load_epoch_num, train_map_dirs, val_map_dirs, train_model_dir, val_model_dir, load_model_path, evl_folder_path = initial_setting(
    model_type, data_path, map_dirs, DEVICE)

model = UNET(in_channels, out_channels).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler()

# Generate data loaders
train_loader, val_loader = get_loaders(train_map_dirs, train_model_dir,
                                        val_map_dirs, val_model_dir,
                                        batch_size, num_workers,
                                        pin_memory)

In [6]:
print(train_map_dirs)

['/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/map_sample']


In [ ]:
temp_sample_path = os.path.join(MAIN_PATH, 'ready_to_train_and_val')
for model_type1 in MODEL_TYPES:
    temp_sample_path1 = os.path.join(temp_sample_path, f'{model_type1}_pred_prob')
    os.makedirs(temp_sample_path1, exist_ok=True)

pre_map_dirs = val_map_dirs
map_names = [sorted(os.listdir(map_dir)) for map_dir in pre_map_dirs]

# print(map_names[0])
# print(map_names[1])


model.eval()
with torch.no_grad():

    count = 0

    for index in range(len(map_names[0])):
        count+=1
        print(f'{count}/{len(map_names[0])}')
        # for index in range(2):
        for i in range(len(pre_map_dirs)):
            map_dir = pre_map_dirs[i]
            map_name = map_names[i][index]
            map_path = os.path.join(map_dir, map_name)
            map_part = np.load(map_path)

            # concatenate maps in different features
            if i == 0:
                # add a dimension of feature to original map
                map = np.expand_dims(map_part, axis=0)
            else:
                map = np.concatenate((map, map_part), axis=0)
            map = np.expand_dims(map, axis=0)

        # Move map to the specified device and perform prediction
        map = torch.from_numpy(map)
        # print(map.shape)
        map = map.to(DEVICE)
        pred_prob = torch.softmax(model(map), 1)
        # print(pred_prob.shape)
        # writen = torch.cat((map[0, :, :, :, :], pred_prob[0, :, :, :, :]), dim=0)
        writen = pred_prob[0, :, :, :, :].cpu().numpy()

        # print(writen.shape)

        file_name = f"{model_type}_pred_prob.{map_name.split('.')[-2]}.npy"
        # print(file_name)
        file_path = os.path.join(temp_sample_path, f'{model_type}_pred_prob',
                                 file_name)
        np.save(file_path, writen)

In [7]:
# split data into training and testing dataset
import splitfolders
import shutil


sample_path = os.path.join(MAIN_PATH, "train_val_data")
# sample_path = os.path.join(HOME_PATH, "TEST", "train_val_data", "test")

os.makedirs(sample_path, exist_ok=True)
splitfolders.ratio(input=temp_sample_path,
                   output=sample_path,
                   seed=44,
                   ratio=(.8, .2),
                   group_prefix=None,
                   move=True)
shutil.rmtree(temp_sample_path)

Copying files: 28899 files [00:00, 43705.95 files/s]


In [ ]:
# test generatred data
import numpy as np
import matplotlib.pyplot as plt

for num in range(0, 5):
    model_type = 'map_sample'
    mapname = f'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/{model_type}/map.{num}.npy'
    aaa = np.load(mapname)
    # print(np.max(aaa))
    plt.imshow(aaa[:, :, 30], cmap='gray', origin='lower')
    plt.title(f'{model_type}.{num}')
    plt.show()

    model_type = 'secondary_strctures'
    filename = f'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/{model_type}/model_{model_type}.{num}.npy'
    aaa = np.load(filename)
    plt.imshow(aaa[:, :, 30], origin='lower')
    plt.title(f'{model_type}.{num}')
    plt.show()

    model_type = 'secondary_strctures_pred_prob'
    filename = f'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/{model_type}/{model_type}.{num}.npy'
    aaa = np.load(filename)
    plt.imshow(aaa[:, :, 30], origin='lower')
    plt.title(f'{model_type}.{num}')
    plt.show()
    print("###")

In [ ]:
def predict_location(map_data, model, out_channels, device):

    new_shape_map, insert_idx = map_reshape(map_data)
    new_shape = new_shape_map.shape
    new_shape_map = torch.from_numpy(new_shape_map)
    new_shape_model = torch.zeros((out_channels, ) + new_shape)

    model.eval()
    # Iterate over 3D cubes of the map data
    with torch.no_grad():
        cube_size = 50
        num_cubes = (new_shape[0] // cube_size) * (
            new_shape[1] // cube_size) * (new_shape[2] // cube_size)
        num_completed = 0
        for zz in range(new_shape[0] // cube_size):
            for yy in range(new_shape[1] // cube_size):
                for xx in range(new_shape[2] // cube_size):
                    # Extract a 3D cube from the map data
                    cube = new_shape_map[None, None,
                                         cube_size * zz:64 + cube_size * zz,
                                         cube_size * yy:64 + cube_size * yy,
                                         cube_size * xx:64 + cube_size * xx]

                    # Move the cube to the specified device and perform prediction
                    cube = cube.to(device)
                    pred_data = torch.softmax(model(cube), 1)

                    # Write predicted data to new_shape_model
                    new_shape_model[:, 7 + cube_size * zz:57 + cube_size * zz,
                                    7 + cube_size * yy:57 + cube_size * yy,
                                    7 + cube_size * xx:57 +
                                    cube_size * xx] = pred_data[0, :, 7:57,
                                                                7:57, 7:57]

                    # Update progress
                    num_completed += 1
                    print(
                        f"{num_completed}/{num_cubes} of prediction is done.",
                        end='\r')
        # print("\n")

    # Extract model data
    part_possibility = new_shape_model[:, insert_idx[0, 0]:insert_idx[0, 1],
                                       insert_idx[1, 0]:insert_idx[1, 1],
                                       insert_idx[2, 0]:insert_idx[2,
                                                                   1]].numpy()

    return part_possibility